## hallo 

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
data_transit = "/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/transit_stations.csv"

df_transit = pd.read_csv(data_transit)

df_transit

In [ ]:
data_sales = "/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/secondary_sales.csv"

df_sales = pd.read_csv(data_sales)

df_sales

In [ ]:
data_rentals = "/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/rentals.csv"

df_rentals = pd.read_csv(data_rentals)

df_rentals['building_era'].count

In [ ]:
data_new= "/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/new_construction.csv"

df_new = pd.read_csv(data_new)

df_new.columns


In [ ]:
data_monthly= "/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/kiez_prices_monthly.csv"

df_monthly= pd.read_csv(data_monthly)

df_monthly

## Analysis Sales Outliers & Missing Values

In [ ]:
####Import
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr

In [ ]:
df_sales.isnull().sum()                

In [ ]:
print(f" {df_sales.duplicated().sum()} duplicates")

In [ ]:
target = 'price_eur' 
round(df_sales[target].describe(),0)

In [ ]:
# Create box plots for all numeric columns
n_cols = len(numeric_cols)
n_rows = (n_cols + 2) // 3  # 3 plots per row
fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    
    # Box plot
    bp = ax.boxplot(df_sales[col].dropna(), vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', alpha=0.7),
                     whiskerprops=dict(color='black', linewidth=1.5),
                     capprops=dict(color='black', linewidth=1.5),
                     medianprops=dict(color='red', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.5))
    
    ax.set_ylabel(col, fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(f'{col}\n(Outliers: {outlier_df[outlier_df["Column"]==col]["Outlier_Percent"].values[0]:.1f}%)',
                 fontsize=10)

# Remove extra subplots
for idx in range(n_cols, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Box Plots — Outlier Detection (Red = Outliers)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('02_boxplots_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
n_rows = (n_cols + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4*n_rows))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    
    # Get bounds
    Q1 = df_sales[col].quantile(0.25)
    Q3 = df_sales[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Histogram
    ax.hist(df_sales[col].dropna(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    
    # Mark bounds
    ax.axvline(lower_bound, color='red', linestyle='--', linewidth=2, label=f'Lower bound: {lower_bound:.2f}')
    ax.axvline(upper_bound, color='red', linestyle='--', linewidth=2, label=f'Upper bound: {upper_bound:.2f}')
    ax.axvline(df_sales[col].median(), color='green', linestyle='-', linewidth=2, label=f'Median: {df_sales[col].median():.2f}')
    
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

# Remove extra subplots
for idx in range(n_cols, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Histograms with IQR Bounds (Red = Outlier Boundaries)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('03_histograms_outlier_bounds.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: 03_histograms_outlier_bounds.png")
plt.show()

In [ ]:
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Count: {df_sales[col].count()}")
    print(f"  Mean: {df_sales[col].mean():.2f}")
    print(f"  Median: {df_sales[col].median():.2f}")
    print(f"  Std: {df_sales[col].std():.2f}")
    print(f"  Min: {df_sales[col].min():.2f}")
    print(f"  Max: {df_sales[col].max():.2f}")
    
    # Skewness & Kurtosis
    skewness = stats.skew(df_sales[col].dropna())
    kurtosis = stats.kurtosis(df_sales[col].dropna())
    print(f"  Skewness: {skewness:.2f}")
    print(f"  Kurtosis: {kurtosis:.2f}")


## Analysis Sales Relationships

In [ ]:
# Select numeric columns + target
corr_matrix = df_sales[numeric_cols + [target]].corr()

# Plot heatmap
plt.figure(figsize=(12, 10))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Heatmap: Numeric Features", fontsize=14, fontweight="bold")
plt.tight_layout()

plt.savefig("05_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
target = 'area_m2' 
round(df_sales[target].describe(),0)

In [ ]:
df_sales.columns

## Sales Edit

In [ ]:
# Assuming df_sales is already loaded
# Create a copy for log-transformed data
df_sales_log = df_sales.copy()

# Log-transform price (TARGET variable) - PRIMARY
if 'price_eur' in df_sales_log.columns:
    df_sales_log['price_eur_log'] = np.log1p(df_sales_log['price_eur'])
    print(f"\n✓ Created: price_eur_log")
    print(f"  Original skewness: {df_sales_log['price_eur'].skew():.3f}")
    print(f"  Log skewness: {df_sales_log['price_eur_log'].skew():.3f}")
    print(f"  → Reduction: {df_sales_log['price_eur'].skew() - df_sales_log['price_eur_log'].skew():.3f}")

# Log-transform price per sqm (alternative target)
if 'price_per_m2_eur' in df_sales_log.columns:
    df_sales_log['price_per_m2_eur_log'] = np.log1p(df_sales_log['price_per_m2_eur'])
    print(f"\n✓ Created: price_per_m2_eur_log")
    print(f"  Original skewness: {df_sales_log['price_per_m2_eur'].skew():.3f}")
    print(f"  Log skewness: {df_sales_log['price_per_m2_eur_log'].skew():.3f}")
    print(f"  → Reduction: {df_sales_log['price_per_m2_eur'].skew() - df_sales_log['price_per_m2_eur_log'].skew():.3f}")

# Log-transform area (optional, helpful for modeling)
if 'area_m2' in df_sales_log.columns:
    df_sales_log['area_m2_log'] = np.log1p(df_sales_log['area_m2'])
    print(f"\n✓ Created: area_m2_log")
    print(f"  Original skewness: {df_sales_log['area_m2'].skew():.3f}")
    print(f"  Log skewness: {df_sales_log['area_m2_log'].skew():.3f}")
    print(f"  → Reduction: {df_sales_log['area_m2'].skew() - df_sales_log['area_m2_log'].skew():.3f}")

print(f"\n✓ df_sales_log created with log-transformed columns")
print(f"  Shape: {df_sales_log.shape}")

In [ ]:
# Create box plots for the three log-transformed columns
log_cols = ['price_eur_log', 'price_per_m2_eur_log', 'area_m2_log']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

for idx, col in enumerate(log_cols):
    ax = axes[idx]
    
    # Box plot
    bp = ax.boxplot(df_sales_log[col].dropna(), vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightgreen', alpha=0.7),
                     whiskerprops=dict(color='black', linewidth=1.5),
                     capprops=dict(color='black', linewidth=1.5),
                     medianprops=dict(color='red', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='red', markersize=6, alpha=0.5))
    
    ax.set_ylabel(col, fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(f'{col}\n(Skewness: {df_sales_log[col].skew():.3f})', fontsize=10)

plt.suptitle('Box Plots — Log-Transformed Features', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('09_boxplots_log_transformed.png', dpi=300, bbox_inches='tight')
plt.show()

## Sales Eras & Floor & USD final df: df_sales_clean 

In [ ]:
def year_to_era(year):
    if pd.isna(year):
        return 'unknown'
    elif year < 1949:
        return 'altbau_pre_1949'
    elif 1949 <= year < 1990:
        return 'post_war_1949_1990'
    elif 1990 <= year < 2010:
        return 'modern_1990_2010'
    else:
        return 'new_post_2010'

df_sales_log['building_era'] = df_sales_log['year_built'].apply(year_to_era)

In [ ]:
# Top floor & Ground floor
df_sales_log['is_top_floor'] = (df_sales_log['floor'] == df_sales_log['total_floors']).astype(int)
df_sales_log['is_ground_floor'] = (df_sales_log['floor'] == 0).astype(int)

In [ ]:
df_sales_log.head()

In [471]:
from sklearn.preprocessing import OrdinalEncoder

# Define ordinal mappings
energy_order = ['A+', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
condition_order = ['saniert', 'modernisiert', 'renoviert', 'renovierungsbedürftig']

# ============================================================================
# ONE-HOT ENCODE: era, height, position
# ============================================================================

era_dummies = pd.get_dummies(df_sales_log['building_era'], prefix='building_era', drop_first=True)
position_dummies = pd.get_dummies(df_sales_log['position'], prefix='position', drop_first=True)

df_sales_log = pd.concat([df_sales_log, era_dummies, position_dummies], axis=1)

print(f"✓ One-hot encoded: building_era, building_height, position")

# ============================================================================
# ORDINAL ENCODE: energy_class, condition
# ============================================================================

# Ordinal encode energy_class
energy_encoder = OrdinalEncoder(categories=[energy_order], handle_unknown='use_encoded_value', unknown_value=-1)
df_sales_log['energy_class_ordinal'] = energy_encoder.fit_transform(df_sales_log[['energy_class']])

# Ordinal encode condition
condition_encoder = OrdinalEncoder(categories=[condition_order], handle_unknown='use_encoded_value', unknown_value=-1)
df_sales_log['condition_ordinal'] = condition_encoder.fit_transform(df_sales_log[['condition']])

print(f"✓ Ordinal encoded: energy_class, condition")

# ============================================================================
# DROP ORIGINAL CATEGORICAL COLUMNS (no longer needed)
# ============================================================================

df_sales_log = df_sales_log.drop(columns=['building_era', 'position', 'energy_class', 'condition'])

print(f"✓ Dropped original categorical columns")
print(f"\nFinal shape: {df_sales_log.shape}")

KeyError: 'building_era'

In [ ]:
df_sales_log.head()

In [ ]:
usd_cols = [col for col in df_sales_log.columns if 'usd' in col.lower()]
if usd_cols:
    df_sales_log = df_sales_log.drop(columns=usd_cols)
    print(f"\n✓ Dropped USD columns: {usd_cols}")

In [ ]:
df_sales_clean = df_sales_log

In [ ]:
df_sales_clean 

## Final DF

In [ ]:
df_sales_clean.to_csv('/Users/lukasstraehnz/code/MarieKatha/berlin-property-intelligence/raw_data/df_sales_clean.csv', index=False)

## Modelling: DBSCAN

In [ ]:
df_sales_clean

## Modelling: Neural Network

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
df_sales_clean

In [ ]:
###Select y
target = 'price_eur_log'
y = df_sales_clean[target].values


###Select X
exclude_cols = [
    'id', 'date_listed',  # Identifiers
    'price_eur', 'price_per_m2_eur', 'price_per_m2_eur_log',  # Leakage/redundant
    'transit_station', 'transit_line', 'transit_distance_type',  # Use distance_min
    'ortsteil', 'bezirk', 'kiez_premium','property_type', 'building_era', 'energy_class', 'position','condition','area_m2',  # Geographic (use distance + location flags)
]


feature_cols = [col for col in df_sales_clean.columns if col not in exclude_cols and col != target]
 
X = df_sales_clean[feature_cols].values

In [ ]:
bool_cols = df_sales_clean[feature_cols].select_dtypes(include=['bool']).columns.tolist()

In [ ]:
###Train test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Identify continuous features
continuous_features = [
    'lat','lon','rooms', 'area_m2', 'floor', 'total_floors', 'year_built',
    'transit_distance_min', 'to_brandenburg_gate_km',
    'price_eur_log', 'price_per_m2_eur_log', 'area_m2_log',
    'property_age', 'rooms_per_sqm',
    'mortgage_rate_at_listing'
]

# Get continuous feature indices
continuous_indices = [feature_cols.index(col) for col in continuous_features if col in feature_cols]

# Scale ONLY continuous features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[:, continuous_indices] = scaler.fit_transform(X_train[:, continuous_indices])
X_test_scaled[:, continuous_indices] = scaler.transform(X_test[:, continuous_indices])

# convert to float32
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# Also convert y
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

##Turn bool to no
bool_cols = df_sales_clean[feature_cols].select_dtypes(include=['bool']).columns.tolist()
df_sales_clean[bool_cols] = df_sales_clean[bool_cols].astype(int)

In [ ]:
###Model
n_features = X_train_scaled.shape[1]
 
model = keras.Sequential([
    # Input layer
    layers.Input(shape=(n_features,)),
    
    # Hidden layer 1
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 2
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Hidden layer 3
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Hidden layer 4
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.1),
    
    # Output layer (regression - no activation)
    layers.Dense(1)
])

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
model.summary()

LR=0.001, BS=16

In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )
    ]
)

In [ ]:
# Predictions
y_train_pred = model.predict(X_train_scaled, verbose=0).flatten()
y_test_pred = model.predict(X_test_scaled, verbose=0).flatten()
 
# Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [ ]:
# Convert from log scale to original € scale
y_train_pred_eur = np.expm1(y_train_pred)
y_test_pred_eur = np.expm1(y_test_pred)
y_train_eur = np.expm1(y_train)
y_test_eur = np.expm1(y_test)
 
test_mae_eur = mean_absolute_error(y_test_eur, y_test_pred_eur)
test_rmse_eur = np.sqrt(mean_squared_error(y_test_eur, y_test_pred_eur))

In [ ]:
print(f"\n" + "="*80)
print("VISUALIZING RESULTS")
print("="*80)
 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
 
# Plot 1: Training history - Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
 
# Plot 2: Training history - MAE
axes[0, 1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[0, 1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[0, 1].set_title('Model MAE', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)
 
# Plot 3: Predictions vs Actual (Test Set, Log Scale)
axes[1, 0].scatter(y_test, y_test_pred, alpha=0.5, s=20)
axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1, 0].set_xlabel('Actual Log(Price)', fontweight='bold')
axes[1, 0].set_ylabel('Predicted Log(Price)', fontweight='bold')
axes[1, 0].set_title(f'Test Set Predictions (R² = {test_r2:.3f})', fontweight='bold', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)
 
# Plot 4: Residuals
residuals = y_test - y_test_pred
axes[1, 1].scatter(y_test_pred, residuals, alpha=0.5, s=20)
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted Log(Price)', fontweight='bold')
axes[1, 1].set_ylabel('Residuals', fontweight='bold')
axes[1, 1].set_title('Residual Plot (Test Set)', fontweight='bold', fontsize=12)
axes[1, 1].grid(alpha=0.3)
 
plt.suptitle('Neural Network Performance', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('neural_network_results.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: neural_network_results.png")
plt.show()

In [ ]:
# Convert train predictions from log scale to € scale
y_train_pred_eur = np.expm1(y_train_pred)
y_train_eur = np.expm1(y_train)

# Calculate MAE and RMSE in € scale
train_mae_eur = mean_absolute_error(y_train_eur, y_train_pred_eur)
train_rmse_eur = np.sqrt(mean_squared_error(y_train_eur, y_train_pred_eur))

# Now create table
results_df = pd.DataFrame({
    'Metric': ['R²', 'MAE (€)', 'RMSE (€)'],
    'Train': [f'{train_r2:.4f}', f'{train_mae_eur:,.0f}', f'{train_rmse_eur:,.0f}'],
    'Test': [f'{test_r2:.4f}', f'{test_mae_eur:,.0f}', f'{test_rmse_eur:,.0f}']
})

print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(results_df.to_string(index=False))
print("="*50)

## Hyper Parameter

import itertools
from sklearn.model_selection import ParameterGrid
from tensorflow.keras import layers as keras_layers

param_grid = {
    'layer_units': [
        [128, 64, 32, 16],
        [256, 128, 64, 32],a
        [64, 32, 16],
        [128, 128, 64, 32],
    ],
    'dropout_rates': [
        [0.3, 0.3, 0.2, 0.1],
        [0.2, 0.2, 0.1, 0.05],
        [0.4, 0.4, 0.3, 0.2],
    ],
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [16, 32, 64],
}

results = []

print("="*80)
print("GRID SEARCH - HYPERPARAMETER OPTIMIZATION")
print("="*80)

for i, params in enumerate(ParameterGrid(param_grid)):
    print(f"\n[{i+1}] Testing: LR={params['learning_rate']}, BS={params['batch_size']}")
    
    layer_units = params['layer_units']
    dropout = params['dropout_rates']
    lr = params['learning_rate']
    bs = params['batch_size']
    
    # Build model
    model = keras.Sequential()
    model.add(keras_layers.Input(shape=(X_train_scaled.shape[1],)))
    
    for j, units in enumerate(layer_units):
        model.add(keras_layers.Dense(units, activation='relu'))
        model.add(keras_layers.BatchNormalization())
        model.add(keras_layers.Dropout(dropout[j]))
    
    model.add(keras_layers.Dense(1))
    
    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )
    
    # Train
    history = model.fit(
        X_train_scaled, y_train,
        epochs=100,
        batch_size=bs,
        validation_split=0.2,
        verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True
            )
        ]
    )
    
    # Evaluate
    y_pred = model.predict(X_test_scaled, verbose=0).flatten()
    test_r2 = r2_score(y_test, y_pred)
    test_mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
    test_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
    
    results.append({
        'Layers': str(layer_units),
        'Dropout': str(dropout),
        'LR': lr,
        'Batch Size': bs,
        'R²': test_r2,
        'MAE': test_mae,
        'RMSE': test_rmse,
    })
    
    print(f"  R²: {test_r2:.4f} | MAE: €{test_mae:,.0f}")

# Results
results_df = pd.DataFrame(results).sort_values('R²', ascending=False)

print(f"\n" + "="*80)
print("TOP 5 CONFIGURATIONS")
print("="*80)
print(results_df.head(5).to_string(index=False))

results_df.to_csv('grid_search_results.csv', index=False)
print(f"\n✓ Saved: grid_search_results.csv")

## One hot ecode & Ordinal 

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Define ordinal mappings
energy_order = ['A+', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
condition_order = ['saniert', 'modernisiert', 'renoviert', 'renovierungsbedürftig']

# ============================================================================
# ONE-HOT ENCODE: era, height, position
# ============================================================================

era_dummies = pd.get_dummies(df_sales_log['building_era'], prefix='building_era', drop_first=True)

position_dummies = pd.get_dummies(df_sales_log['position'], prefix='position', drop_first=True)

df_sales_log = pd.concat([df_sales_log, era_dummies, height_dummies, position_dummies], axis=1)

print(f"✓ One-hot encoded: building_era, building_height, position")

# ============================================================================
# ORDINAL ENCODE: energy_class, condition
# ============================================================================

# Ordinal encode energy_class
energy_encoder = OrdinalEncoder(categories=[energy_order], handle_unknown='use_encoded_value', unknown_value=-1)
df_sales_log['energy_class_ordinal'] = energy_encoder.fit_transform(df_sales_log[['energy_class']])

# Ordinal encode condition
condition_encoder = OrdinalEncoder(categories=[condition_order], handle_unknown='use_encoded_value', unknown_value=-1)
df_sales_log['condition_ordinal'] = condition_encoder.fit_transform(df_sales_log[['condition']])

print(f"✓ Ordinal encoded: energy_class, condition")

# ============================================================================
# DROP ORIGINAL CATEGORICAL COLUMNS (no longer needed)
# ============================================================================

df_sales_log = df_sales_log.drop(columns=['building_era', 'position', 'energy_class', 'condition'])

print(f"✓ Dropped original categorical columns")
print(f"\nFinal shape: {df_sales_log.shape}")

In [470]:
df_sales_clean.columns()

TypeError: 'Index' object is not callable